In [66]:
import pandas as pd
import numpy as np
from MyTfIdfVectorizer import MyTfIdfVectorizer 
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import re


# Part E: Kiểm thử  trên corpus nhỏ và so sánh với TfidfVectorizer của sklearn

In [67]:
data = pd.DataFrame({
    "text": [
        "cat eats fish",
        "dog eats fish",
        "cat likes fish"
    ]
})

my_vectorizer = MyTfIdfVectorizer(data)
sklearn_vectorizer = TfidfVectorizer()

## 0.1 My Tf-Idf

In [68]:
my_vectorizer.buildVocabulary("text")
expected_vocab = [
    "cat",
    "dog",
    "eats",
    "fish",
    "likes"
]

assert my_vectorizer.vocab == expected_vocab


In [69]:
my_vectorizer.compute_counts("text")

expected_counts = np.array([
    [1, 0, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [1, 0, 0, 1, 1]
], dtype=np.float32)

assert np.array_equal(
    my_vectorizer.count_matrix,
    expected_counts
)

In [70]:
# compute_tf
my_vectorizer.compute_tf()

expected_tf = np.array([
    [1, 0,   1, 1, 0],
    [0,   1, 1, 1, 0],
    [1, 0,   0,   1, 1]
], dtype=np.float32)

assert np.allclose(
    my_vectorizer.tf,
    expected_tf,
    atol=1e-9
)

In [71]:
# compute_df
my_vectorizer.compute_df()

expected_df = np.array([2, 1, 2, 3, 1])

assert np.array_equal(
    my_vectorizer.df,
    expected_df
)


In [72]:
# compute idf
my_vectorizer.compute_idf()

expected_idf = np.log(
    4 / (1 + expected_df)
) + 1

assert np.allclose(
    my_vectorizer.idf,
    expected_idf,
    atol=1e-9
)


In [73]:
# compute_tfidf

my_vectorizer.compute_tfidf()

raw_tfidf = expected_tf * expected_idf

norms = np.linalg.norm(
    raw_tfidf,
    axis=1,
    keepdims=True
)

expected_tfidf = raw_tfidf / norms
print(expected_tfidf)
print(my_vectorizer.tfidf)
assert np.allclose(
    my_vectorizer.tfidf,
    expected_tfidf,
    atol=1e-9
)

[[0.61980538 0.         0.61980538 0.48133417 0.        ]
 [0.         0.72033345 0.54783215 0.42544054 0.        ]
 [0.54783215 0.         0.         0.42544054 0.72033345]]
[[0.61980538 0.         0.61980538 0.48133417 0.        ]
 [0.         0.72033345 0.54783215 0.42544054 0.        ]
 [0.54783215 0.         0.         0.42544054 0.72033345]]


In [74]:
#compute_cosine_similarity
my_vectorizer.compute_cosine_similarity()

# Test diagonal = 1
assert np.allclose(
    np.diag(my_vectorizer.cosine_similarity),
    np.ones(3),
    atol=1e-9
)

# Test symmetry
assert np.allclose(
    my_vectorizer.cosine_similarity,
    my_vectorizer.cosine_similarity.T,
    atol=1e-9
)

## 0.2 Compare with sklearn TfIdfVectorizer

In [75]:
sklearn_vectorizer = TfidfVectorizer(
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False
)
cv = CountVectorizer()
count_matrix = cv.fit_transform(data["text"])

row_sums = np.asarray(count_matrix.sum(axis=1)).ravel()

sklearn_tf = count_matrix.multiply(
    1 / row_sums[:, None]
)
sklearn_tfidf = sklearn_vectorizer.fit_transform(data["text"])
sklearn_idf = sklearn_vectorizer.idf_

print(type(my_vectorizer.tf))
print(type(my_vectorizer.idf))
print(type(my_vectorizer.tfidf))


<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [76]:
def compare_matrices(name, sklearn_result, my_result, atol=1e-6):

    if hasattr(sklearn_result, "toarray"):
        sklearn_result = sklearn_result.toarray()

    print(f"{name}:")
    print("  Shape:", sklearn_result.shape, my_result.shape)

    print(
        "  Equal:",
        np.allclose(
            sklearn_result,
            my_result,
            atol=atol
        )
    )

    print(
        "  Max error:",
        np.max(
            np.abs(
                sklearn_result - my_result
            )
        )
    )

In [77]:
compare_matrices(
    "TF",
    sklearn_tf,
    my_vectorizer.tf
)

compare_matrices(
    "IDF",
    sklearn_idf,
    my_vectorizer.idf
)

compare_matrices(
    "TF-IDF",
    sklearn_tfidf,
    my_vectorizer.tfidf
)


TF:
  Shape: (3, 5) (3, 5)
  Equal: False
  Max error: 0.6666666666666667
IDF:
  Shape: (5,) (5,)
  Equal: True
  Max error: 0.0
TF-IDF:
  Shape: (3, 5) (3, 5)
  Equal: True
  Max error: 0.0


# Part F: Preprocessing Ablation

In [78]:
df = pd.read_json("/home/vitquay1708/Study_Space/NLP/lab1/c4-train.00000-of-01024-30K.json.gz", lines=True)
df.head(5)

,text,timestamp,url
0,Beginners BBQ Class Taking Place in Missoula!\...,2019-04-25 12:57:54+00:00,https://klyq.com/beginners-bbq-class-taking-pl...
1,Discussion in 'Mac OS X Lion (10.7)' started b...,2019-04-21 10:07:13+00:00,https://forums.macrumors.com/threads/restore-f...
2,Foil plaid lycra and spandex shortall with met...,2019-04-25 10:40:23+00:00,https://awishcometrue.com/Catalogs/Clearance/T...
3,How many backlinks per day for new site?\nDisc...,2019-04-21 12:46:19+00:00,https://www.blackhatworld.com/seo/how-many-bac...
4,The Denver Board of Education opened the 2017-...,2019-04-20 14:33:21+00:00,http://bond.dpsk12.org/category/news/


## 1. Minimal

In [79]:
df['text_lower'] = df['text'].str.lower()
df['pipeline_1'] = df['text_lower'].apply(lambda x: x.split())
display(df[['text', 'text_lower', 'pipeline_1']].head(3))


,text,text_lower,pipeline_1
0,Beginners BBQ Class Taking Place in Missoula!\...,beginners bbq class taking place in missoula!\...,"[beginners, bbq, class, taking, place, in, mis..."
1,Discussion in 'Mac OS X Lion (10.7)' started b...,discussion in 'mac os x lion (10.7)' started b...,"[discussion, in, 'mac, os, x, lion, (10.7)', s..."
2,Foil plaid lycra and spandex shortall with met...,foil plaid lycra and spandex shortall with met...,"[foil, plaid, lycra, and, spandex, shortall, w..."


## 2. Pipeline B - Normalized

In [80]:
df['pipeline_2'] = df['text_lower'].apply(lambda x: re.findall(r'\b\w+\b', x))
display(df[['text', 'text_lower', 'pipeline_2']].head(3))

,text,text_lower,pipeline_2
0,Beginners BBQ Class Taking Place in Missoula!\...,beginners bbq class taking place in missoula!\...,"[beginners, bbq, class, taking, place, in, mis..."
1,Discussion in 'Mac OS X Lion (10.7)' started b...,discussion in 'mac os x lion (10.7)' started b...,"[discussion, in, mac, os, x, lion, 10, 7, star..."
2,Foil plaid lycra and spandex shortall with met...,foil plaid lycra and spandex shortall with met...,"[foil, plaid, lycra, and, spandex, shortall, w..."


## 3 Pipeline C - Extended

In [81]:
from transformers import AutoTokenizer

subword_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

df['pipeline_3'] = df['text_lower'].apply(lambda x: subword_tokenizer.tokenize(x))
display(df[['text', 'text_lower', 'pipeline_3']].head(3))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2531 > 512). Running this sequence through the model will result in indexing errors


,text,text_lower,pipeline_3
0,Beginners BBQ Class Taking Place in Missoula!\...,beginners bbq class taking place in missoula!\...,"[begin, ##ners, bb, ##q, class, taking, place,..."
1,Discussion in 'Mac OS X Lion (10.7)' started b...,discussion in 'mac os x lion (10.7)' started b...,"[discussion, in, ', mac, os, x, lion, (, 10, ...."
2,Foil plaid lycra and spandex shortall with met...,foil plaid lycra and spandex shortall with met...,"[foil, plaid, l, ##y, ##cr, ##a, and, span, ##..."


## Result Comparison

In [82]:
from collections import Counter

def tokenizer1(text):
    return text.split()

def tokenizer2(text):
    return re.findall(r'\b\w+\b', text)

def tokenizer3(text):
    return subword_tokenizer.tokenize(text)

tokenizers = {'pipeline_1':tokenizer1, 'pipeline_2':tokenizer2, 'pipeline_3':tokenizer3}
def calculate_oov_rate(tokenized_docs, vocab):
    total = 0
    oov = 0
    for doc in tokenized_docs:
        for token in doc:
            total += 1
            if token not in vocab:
                oov+=1
    return oov/total if total > 0 else 0

def compare_pipelines(
    df,
    pipeline_columns,
    test_docs,
    tokenizers
):
    results = {}

    for _, column in enumerate(pipeline_columns):

        docs = df[column].dropna().tolist()
        vocabulary = set(
            token
            for doc in docs
            for token in doc
        )
        vocab_size = len(vocabulary)
        avg_tokens = (
            sum(len(doc) for doc in docs) / len(docs)
            if docs else 0
        )
        n_docs = len(docs)

        non_zero = sum(
            len(set(doc))
            for doc in docs
        )

        total_entries = n_docs * vocab_size

        sparsity = (
            1 - non_zero / total_entries
            if total_entries > 0
            else 0
        )

        test_tokenized = [
            tokenizers[column](text)
            for text in test_docs
        ]

        oov_rate = calculate_oov_rate(test_tokenized, vocabulary)

        results[column] = {
            "Vocabulary size": vocab_size,
            "Average tokens/document": avg_tokens,
            "Matrix sparsity": sparsity,
            "OOV rate": oov_rate
        }

    return pd.DataFrame(results).T

In [83]:
test_docs = [
    "The cat is running quickly.",
    "I love machine learning.",
    "This is an amazing example.",
    "Natural language processing is interesting.",
    "The model generates new tokens.",
    "Students are learning artificial intelligence.",
    "The weather is beautiful today.",
    "This sentence contains an unknownword.",
    "Transformers are widely used in NLP.",
    "Subword tokenization handles rare words."
]
result_comparison = compare_pipelines(df, ['pipeline_1', 'pipeline_2', 'pipeline_3'],test_docs,tokenizers)

display(result_comparison)

,Vocabulary size,Average tokens/document,Matrix sparsity,OOV rate
pipeline_1,473388.0,361.089067,0.999611,0.24
pipeline_2,193837.0,369.696367,0.999122,0.24
pipeline_3,28339.0,465.099200,0.993200,0.00


# Part G: Document Search Engine